In [0]:
# ===========================================
# Notebook Name:
# 01_ingest_event_result
#
# Purpose:
# Fetch Pokemon tournament result JSON
# and store the raw response in Bronze Delta.
#
# Input:
# Tournament ID
#
# Output:
# pokemon.bronze.event_result_raw
# pokemon.bronze.scrape_log
# ===========================================

In [0]:
CATALOG_NAME = "pokemon"
BRONZE_SCHEMA = "bronze"

EVENT_RESULT_TABLE = (
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.event_result_raw"
)

SCRAPE_LOG_TABLE = (
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.scrape_log"
)

BASE_URL = "https://players.pokemon-card.com"
RESULT_API_URL = f"{BASE_URL}/event_result_detail_search"

TOURNAMENT_ID = "1032136"
PER_PAGE = 100

print("Target tournament:", TOURNAMENT_ID)
print("Output table:", EVENT_RESULT_TABLE)

In [0]:
from __future__ import annotations

from datetime import datetime, timezone
from typing import Any
import hashlib
import json
import time
import uuid

import requests

In [0]:
REQUEST_HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ja,en-US;q=0.9,en;q=0.8",
    "User-Agent": (
        "Mozilla/5.0 "
        "(compatible; PokemonLakehouse/0.1)"
    ),
    "X-Accept-Version": "v1",
}


def fetch_event_result(
    tournament_id: str,
    per_page: int = 100,
    timeout: int = 30,
) -> tuple[dict[str, Any], int, str, int]:

    params = {
        "event_holding_id": tournament_id,
        "offset": 0,
        "per_page": per_page,
    }

    request_headers = {
        **REQUEST_HEADERS,
        "Referer": (
            f"{BASE_URL}/event/detail/"
            f"{tournament_id}/result"
        ),
    }

    started_at = time.perf_counter()

    response = requests.get(
        RESULT_API_URL,
        params=params,
        headers=request_headers,
        timeout=timeout,
    )

    elapsed_ms = int(
        (time.perf_counter() - started_at) * 1000
    )

    response.raise_for_status()

    payload = response.json()

    if payload.get("code") != 200:
        raise RuntimeError(
            f"Unexpected API response: {payload}"
        )

    return (
        payload,
        response.status_code,
        response.url,
        elapsed_ms,
    )

In [0]:
payload, http_status, request_url, elapsed_ms = (
    fetch_event_result(
        tournament_id=TOURNAMENT_ID,
        per_page=PER_PAGE,
    )
)

print("HTTP status:", http_status)
print("Request URL:", request_url)
print("Elapsed:", elapsed_ms, "ms")
print("Result count:", payload.get("count"))

In [0]:
print(
    json.dumps(
        payload,
        ensure_ascii=False,
        indent=2,
    )[:3000]
)

In [0]:
payload_text = json.dumps(
    payload,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
)

response_hash = hashlib.sha256(
    payload_text.encode("utf-8")
).hexdigest()

scraped_at = datetime.now(timezone.utc)

print("Payload length:", len(payload_text))
print("Response hash:", response_hash)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)

event_raw_schema = StructType([
    StructField(
        "tournament_id",
        StringType(),
        False,
    ),
    StructField(
        "source_url",
        StringType(),
        False,
    ),
    StructField(
        "http_status",
        IntegerType(),
        True,
    ),
    StructField(
        "payload",
        StringType(),
        False,
    ),
    StructField(
        "payload_format",
        StringType(),
        False,
    ),
    StructField(
        "response_hash",
        StringType(),
        False,
    ),
    StructField(
        "scraped_at",
        TimestampType(),
        False,
    ),
])

event_raw_row = [{
    "tournament_id": TOURNAMENT_ID,
    "source_url": request_url,
    "http_status": http_status,
    "payload": payload_text,
    "payload_format": "json",
    "response_hash": response_hash,
    "scraped_at": scraped_at,
}]

event_raw_df = (
    spark.createDataFrame(
        event_raw_row,
        schema=event_raw_schema,
    )
    .withColumn(
        "ingestion_date",
        F.to_date("scraped_at"),
    )
)

display(event_raw_df)

In [0]:
existing_count = (
    spark.table(EVENT_RESULT_TABLE)
    .filter(
        (F.col("tournament_id") == TOURNAMENT_ID)
        & (F.col("response_hash") == response_hash)
    )
    .limit(1)
    .count()
)

print("Existing identical rows:", existing_count)

In [0]:
was_inserted = False

if existing_count == 0:
    (
        event_raw_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(EVENT_RESULT_TABLE)
    )

    was_inserted = True
    print("New raw response inserted.")
else:
    print("Identical raw response already exists. Skipped.")

In [0]:
display(
    spark.sql(f"""
    SELECT
        tournament_id,
        http_status,
        payload_format,
        response_hash,
        scraped_at,
        ingestion_date,
        length(payload) AS payload_length
    FROM {EVENT_RESULT_TABLE}
    WHERE tournament_id = '{TOURNAMENT_ID}'
    ORDER BY scraped_at DESC
    """)
)

In [0]:
latest_payload = (
    spark.table(EVENT_RESULT_TABLE)
    .filter(
        F.col("tournament_id") == TOURNAMENT_ID
    )
    .orderBy(
        F.col("scraped_at").desc()
    )
    .select("payload")
    .first()
)

print(
    json.dumps(
        json.loads(latest_payload["payload"]),
        ensure_ascii=False,
        indent=2,
    )[:3000]
)

In [0]:
log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("source_type", StringType(), False),
    StructField("source_id", StringType(), True),
    StructField("request_url", StringType(), False),
    StructField("http_status", IntegerType(), True),
    StructField("status", StringType(), False),
    StructField("elapsed_ms", IntegerType(), True),
    StructField("records_found", IntegerType(), True),
    StructField("error_type", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("scraped_at", TimestampType(), False),
])

log_status = (
    "SUCCESS_INSERTED"
    if was_inserted
    else "SUCCESS_SKIPPED"
)

log_row = [{
    "run_id": str(uuid.uuid4()),
    "source_type": "event_result",
    "source_id": TOURNAMENT_ID,
    "request_url": request_url,
    "http_status": http_status,
    "status": log_status,
    "elapsed_ms": elapsed_ms,
    "records_found": int(payload.get("count", 0)),
    "error_type": None,
    "error_message": None,
    "scraped_at": scraped_at,
}]

log_df = (
    spark.createDataFrame(
        log_row,
        schema=log_schema,
    )
    .withColumn(
        "elapsed_ms",
        F.col("elapsed_ms").cast("long"),
    )
    .withColumn(
        "ingestion_date",
        F.to_date("scraped_at"),
    )
)

(
    log_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(SCRAPE_LOG_TABLE)
)

display(log_df)

In [0]:
display(
    spark.sql(f"""
    DESCRIBE HISTORY {EVENT_RESULT_TABLE}
    """)
)